# Create Unified DB
Creation d'une DB unifiée qui reprenne les infos de chacun des modeles

In [1]:
import sqlite3

In [2]:
unified           = sqlite3.connect("./unified.db")
languages         = sqlite3.connect("./language.db")
llama             = sqlite3.connect("./llama.db")
mistral7b         = sqlite3.connect("./mistral7b.db")
mistralnemo       = sqlite3.connect("./mistralnemo.db")
gpt4omini         = sqlite3.connect("./gpt4o-mini.db")
vieille_france    = sqlite3.connect("./vieille_france.db")

In [3]:
# cursors
c_unified         = unified.cursor()
c_languages       = languages.cursor()
c_llama           = llama.cursor()
c_mistral7b       = mistral7b.cursor()
c_mistralnemo     = mistralnemo.cursor()
c_gpt4omini       = gpt4omini.cursor()
c_vieille_france  = vieille_france.cursor()

In [4]:
# create unified tables
models = """
CREATE TABLE models (
    'name' TEXT PRIMARY KEY);
"""

projects = """
CREATE TABLE projects (
    'name' TEXT PRIMARY KEY);
"""

languages = """
CREATE TABLE languages (
    'name' TEXT PRIMARY KEY);
"""

subsets = """
CREATE TABLE subsets (
    'name' TEXT PRIMARY KEY);
"""

documents = """
CREATE TABLE documents (
    'id'      INT  PRIMARY KEY,
    'text'    TEXT NOT NULL,
    'project' TEXT NOT NULL REFERENCES projects('name'),
    'lang'    TEXT NOT NULL REFERENCES languages('name'),
    'subset'  TEXT NOT NULL REFERENCES subsets('name'),
    'file_id' TEXT NOT NULL
);
"""

labeling = """
CREATE TABLE labeling (
    'id'       INT  NOT NULL REFERENCES documents('id'),
    'model'    TEXT NOT NULL REFERENCES models('name'),
    'labeling' TEXT NOT NULL,
    
    PRIMARY KEY ('id', 'model')
);
"""

ranking = """
CREATE TABLE ranking (
    'id'       INT  NOT NULL REFERENCES documents('id'),
    'model'    TEXT NOT NULL REFERENCES models('name'),
    'usrname'  TEXT NOT NULL,
    'ranking'  INT  NOT NULL DEFAULT 0,

    PRIMARY KEY ('id', 'model', 'usrname'),
    FOREIGN KEY ('id', 'model') REFERENCES labeling('id','model')
);
"""
c_unified.execute(models)
c_unified.execute(projects)
c_unified.execute(languages)
c_unified.execute(subsets)

c_unified.execute(documents)
c_unified.execute(labeling)
c_unified.execute(ranking)

In [5]:
# insert données statiques
c_unified.execute("""
    INSERT INTO projects(name) VALUES
        ('pardons'),
        ('sentences');
""")
c_unified.execute("""
    INSERT INTO languages(name) VALUES
        ('nl'),
        ('fr');
""")
c_unified.execute("""
    INSERT INTO subsets(name) VALUES
        ('train'),
        ('test'),
        ('valid');
""")
c_unified.execute("""
    INSERT INTO models(name) VALUES
        ('gpt4o-mini'),
        ('llama'),
        ('mistral7b'),
        ('mistralnemo'),
        ('vieille_france');
""")

In [10]:
c_languages.execute("select id, project, language, subset, file_id, text from dataset;")
c_unified.executemany("INSERT INTO documents(id, project, lang, subset, file_id, text) VALUES (?, ?, ?, ?, ?, ?)", c_languages.fetchall())

In [5]:
c_gpt4omini.execute("select id, 'gpt4o-mini', labeling from dataset")
c_unified.executemany("INSERT INTO labeling(id, model, labeling) VALUES (?, ?, ?)", c_gpt4omini.fetchall())

In [ ]:
def unify(model, cursor):
    cursor.execute(f"select id, '{model}', labeling from dataset")
    c_unified.executemany("INSERT INTO labeling(id, model, labeling) VALUES (?, ?, ?)", cursor.fetchall())

unify("llama", c_llama)
unify("llama", c_llama)



In [11]:
# cloture du truc
unified           .commit()
llama             .commit()
mistral7b         .commit()
mistralnemo       .commit()
gpt4omini         .commit()
vieille_france    .commit()

In [12]:
# cloture du truc
unified           .close()
llama             .close()
mistral7b         .close()
mistralnemo       .close()
gpt4omini         .close()
vieille_france    .close()